## Setup: Install Dependencies

**RUN THIS FIRST** - Installs lifelines library for Cox Proportional Hazards modeling.

⚠️ **This will restart the kernel** - After completion, run the Configuration cell next to restore variables.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lifelines==0.27.8", "scikit-survival"])
print("Dependencies installed")

## Configuration

In [ ]:
# Configuration
TARGET_ASSETS = [
    'RV2_U2_Boiler',
    'RV3_U3_Steam_Turbine',
    'RV3_U3_Boiler_Feed_Pump_East'
]

HORIZON_DAYS = 14  # Censoring point
# Feature count is determined dynamically by EPV rule (events/10, capped at 20)
# Previously hardcoded to 50, which violated EPV requirement
EXPERIMENT_NAME = "Phase3-LongTerm-Cox-Survival"

# ─── Eventhouse running-state gate ───
KUSTO_URI = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
KUSTO_DB = "pi-realtime-db"

def _kusto_tok():
    try:
        import notebookutils as _n; _c = _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; _c = _m.credentials
    for _a in (KUSTO_URI, "kusto", "pbi"):
        try:
            _t = _c.getToken(_a)
            if _t: return _t
        except Exception:
            pass
    raise RuntimeError("could not acquire Kusto token")

def read_kusto(query):
    return (spark.read
        .format("com.microsoft.kusto.spark.datasource")
        .option("accessToken", _kusto_tok())
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .load())

def check_running_state():
    """Query Eventhouse for current running state of each asset."""
    kql = """
    let rv2=PiEvents|where not(Questionable) and Tag=="RV2:BTPU2BPDRUM.AG"|top 1 by Ts desc|extend asset_id="RV2_U2_Boiler",Threshold=600;
    let rv3=PiEvents|where not(Questionable) and Tag=="RV3:TXSU3TS15A.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Steam_Turbine",Threshold=6;
    let bfp=PiEvents|where not(Questionable) and Tag=="RV3:FPSU3TS24E.AG"|top 1 by Ts desc|extend asset_id="RV3_U3_Boiler_Feed_Pump_East",Threshold=20;
    union rv2,rv3,bfp
    |extend V=toreal(Value)
    |extend IsRunning=(V > Threshold)
    |project asset_id, IsRunning, V=round(V,1), Tag, Ts
    """
    try:
        rs_df = read_kusto(kql)
        status = {}
        for row in rs_df.collect():
            status[row.asset_id] = bool(row.IsRunning)
            state = "RUNNING" if row.IsRunning else "STOPPED"
            print(f"  {row.asset_id}: {state} (value={row.V}, tag={row.Tag})")
        for a in TARGET_ASSETS:
            if a not in status:
                status[a] = True
                print(f"  {a}: UNKNOWN (no data) — assuming RUNNING")
        return status
    except Exception as e:
        print(f"  ⚠️ Running-state check failed: {e} — assuming all RUNNING")
        return {a: True for a in TARGET_ASSETS}

print("="*70)
print("COX PROPORTIONAL HAZARDS SURVIVAL ANALYSIS")
print("="*70)
print(f"\n✔ Target assets: {len(TARGET_ASSETS)}")
print(f"✔ Observation horizon: {HORIZON_DAYS} days")
print(f"✔ MLflow experiment: {EXPERIMENT_NAME}")

print("\n🔍 Checking current asset running state...")
ASSET_RUNNING = check_running_state()
ACTIVE_ASSETS = [a for a in TARGET_ASSETS if ASSET_RUNNING.get(a, True)]
STOPPED_ASSETS = [a for a in TARGET_ASSETS if not ASSET_RUNNING.get(a, True)]
if STOPPED_ASSETS:
    print(f"\n⚠️ Stopped assets (will skip scoring/watchlist): {', '.join(STOPPED_ASSETS)}")
print(f"✔ Active assets for scoring: {', '.join(ACTIVE_ASSETS)}")

## Phase 1b: Daily Label Construction
Build daily survival labels from GADS events for each asset. Writes to `gold.daily_survival_labels`.

In [ ]:
# Phase 1b: Daily Label Construction with days_to_next_stop
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta

HISTORICAL_WINDOW_DAYS = 365  # Use last year of data

print(f"Long-term survival analysis parameters (EXTENDED HORIZON):")
print(f"  Assets: {TARGET_ASSETS}")
print(f"  Prediction horizon: {HORIZON_DAYS} days (90-day outlook)")
print(f"  Historical window: {HISTORICAL_WINDOW_DAYS} days")

# Load GADS stop events (same as Phase 1 short-term)
gads_events = spark.table("gold.fact_gads_event")

# Filter to target assets and FORCED outages only (align with Phase 1 short-term pipeline)
# Exclude: PO (planned), RS (reserve shutdown), D1-D4/D (deratings)
stop_events = gads_events.filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("EVENT_TYPE_CD").isin(['U1', 'U2', 'U3', 'MO', 'SF', 'D1', 'D3'])
).select(
    F.col("asset_id"),
    F.col("REAL_START_DT").cast("date").alias("stop_date"),
    F.col("EVENT_TYPE_CD").alias("event_type"),
    F.col("CAUSE_OF_EVENT").alias("event_description")
).distinct()

print(f"\n✔ Loaded GADS stop events: {stop_events.count():,} events")
stop_events.groupBy("asset_id").count().show()

# Generate daily date grid for each asset (last HISTORICAL_WINDOW_DAYS days)
end_date = datetime.now().date()
start_date = end_date - timedelta(days=HISTORICAL_WINDOW_DAYS)

date_range = spark.range(
    0, HISTORICAL_WINDOW_DAYS + 1
).select(
    F.expr(f"date_add(date('{start_date}'), cast(id as int))").alias("calendar_date")
)

# Cross-join with assets to get daily grid
asset_dates = spark.createDataFrame(
    [(asset,) for asset in TARGET_ASSETS],
    ["asset_id"]
).crossJoin(date_range)

print(f"\n✔ Generated daily grid: {asset_dates.count():,} asset-days")
print(f"  Date range: {start_date} to {end_date}")

# For each asset-day, calculate days_to_next_stop
# Window function: look forward to find next stop
w_future = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(
    Window.currentRow, Window.unboundedFollowing
)

# Join asset-days with stop events
labels_with_stops = asset_dates.join(
    stop_events.select("asset_id", "stop_date"),
    (asset_dates.asset_id == stop_events.asset_id) &
    (stop_events.stop_date >= asset_dates.calendar_date),
    "left"
).select(
    asset_dates.asset_id,
    asset_dates.calendar_date,
    stop_events.stop_date
)

# Calculate days to each future stop, then take minimum (next stop)
labels_with_days = labels_with_stops.withColumn(
    "days_to_stop",
    F.datediff(F.col("stop_date"), F.col("calendar_date"))
).groupBy("asset_id", "calendar_date").agg(
    F.min("days_to_stop").alias("days_to_next_stop")
)

# Cap at HORIZON_DAYS (censored observations)
# NULL means no stop found → censor at 14 days
labels_daily = labels_with_days.withColumn(
    "days_to_next_stop",
    F.when(F.col("days_to_next_stop").isNull(), F.lit(HORIZON_DAYS))
     .when(F.col("days_to_next_stop") > HORIZON_DAYS, F.lit(HORIZON_DAYS))
     .otherwise(F.col("days_to_next_stop"))
).withColumn(
    "censored",
    F.when(F.col("days_to_next_stop") >= HORIZON_DAYS, F.lit(1)).otherwise(F.lit(0))
)

# Add binary labels for intermediate horizons (useful for monitoring)
labels_daily = labels_daily.withColumn("label_stop_7d", 
    F.when(F.col("days_to_next_stop") <= 7, 1).otherwise(0)
).withColumn("label_stop_14d",
    F.when(F.col("days_to_next_stop") <= 14, 1).otherwise(0)
)

# Save to gold schema
labels_daily.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold.daily_survival_labels")

print(f"\n✔ Saved gold.daily_survival_labels: {labels_daily.count():,} rows")
print(f"\nLabel distribution:")
# NOTE: Unit-level label contamination warning
#   RV3_U3_Steam_Turbine and RV3_U3_Boiler_Feed_Pump_East are both mapped to
#   the same physical unit (Unit 87). When the unit trips, both assets receive
#   an identical "event" label on the same day. As a result, the BFP East
#   event count is inflated by turbine-caused stops that the BFP did not
#   actually cause. Downstream feature attribution for the BFP should be
#   interpreted with this confounding in mind. Future work: distinguish
#   asset-specific causes from unit-level trips in the GADS event mapping.
print("\n" + "="*70)
print("⚠️  UNIT-LEVEL LABEL WARNING")
print("="*70)
print("RV3_U3_Steam_Turbine and RV3_U3_Boiler_Feed_Pump_East share Unit 87")
print("labels. Identical event counts/avg_days_to_stop between these two")
print("assets reflect unit-level trips, not asset-specific BFP failures.")
print("BFP East risk drivers should be cross-referenced with turbine state.")
print("="*70)

labels_daily.groupBy("asset_id").agg(
    F.count("*").alias("total_days"),
    F.avg("days_to_next_stop").alias("avg_days_to_stop"),
    F.sum(F.when(F.col("label_stop_7d") == 1, 1).otherwise(0)).alias("stops_within_7d"),
    F.sum(F.when(F.col("label_stop_14d") == 1, 1).otherwise(0)).alias("stops_within_14d"),
    F.sum("censored").alias("censored_days")
).show(truncate=False)

# Preview sample
print("\nSample labels (recent days):")
labels_daily.filter(F.col("calendar_date") >= F.date_sub(F.current_date(), 30)) \
    .orderBy("asset_id", F.desc("calendar_date")).show(20, truncate=False)

## Phase 2b: Daily Feature Engineering
Pivot PI sensor data into daily statistics per asset. Writes to `ml.training_longterm`.

In [ ]:
# Phase 2b: Daily Feature Engineering
from pyspark.sql import Window
from pyspark.sql import functions as F

# Load PI fact table and tag mapping
pi_fact = spark.table("gold.fact_pi")
tag_mapping = spark.table("gold.bridge_pi_tag_to_asset")

# Filter to target assets
asset_tags = tag_mapping.filter(F.col("asset_id").isin(TARGET_ASSETS))

# Join PI data with asset mapping
pi_asset = pi_fact.join(
    asset_tags.select(F.col("Tag"), F.col("asset_id").alias("bridge_asset_id")),
    "Tag",
    "inner"
).select(
    F.coalesce(F.col("asset_id"), F.col("bridge_asset_id")).alias("asset_id"),
    "Tag",
    "Timestamp",
    "ValueNumeric"
)

print(f"✔ Loaded PI data: {pi_asset.count():,} rows")
print(f"✔ Tags per asset:")
asset_tags.groupBy("asset_id").count().show(truncate=False)

# Convert timestamp to date for daily aggregation
pi_daily_base = pi_asset.withColumn(
    "calendar_date",
    F.to_date(F.col("Timestamp"))
).select(
    "asset_id",
    "Tag", 
    "calendar_date",
    "ValueNumeric"
)

# 1. Daily Statistics per tag (min/max/mean/std)
print("\n1. Computing daily statistics per tag...")

daily_stats = pi_daily_base.groupBy("asset_id", "calendar_date", "Tag").agg(
    F.min("ValueNumeric").alias("daily_min"),
    F.max("ValueNumeric").alias("daily_max"),
    F.avg("ValueNumeric").alias("daily_mean"),
    F.stddev("ValueNumeric").alias("daily_std"),
    F.count("ValueNumeric").alias("daily_count")
)

print(f"   ✔ Daily stats: {daily_stats.count():,} asset-day-tag rows")

# Pivot to wide format (each tag becomes 4 columns: min/max/mean/std)
# First, get distinct tags
distinct_tags = asset_tags.select("Tag").distinct().collect()
tag_list_raw = [row.Tag for row in distinct_tags]

# Sanitize tag names to avoid dots/colons in pivoted column names
import re
tag_list = [re.sub(r'[^a-zA-Z0-9_]', '_', t) for t in tag_list_raw]
safe_to_original = {re.sub(r'[^a-zA-Z0-9_]', '_', t): t for t in tag_list_raw}  # B8: exact SQL lookups
tag_list.sort()  # B4 fix: deterministic ordering
daily_stats = daily_stats.withColumn("safe_tag", F.regexp_replace(F.col("Tag"), "[^a-zA-Z0-9_]", "_"))

print(f"   ✔ Pivoting {len(tag_list)} tags (sanitized names)...")

# Create separate dataframes for each statistic type (pivot on safe_tag)
daily_min_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_min"))
daily_max_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_max"))
daily_mean_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_mean"))
daily_std_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_std"))

# Rename columns with suffixes (tag names already sanitized)
for tag in tag_list:
    daily_min_pivot = daily_min_pivot.withColumnRenamed(tag, f"{tag}_daily_min")
    daily_max_pivot = daily_max_pivot.withColumnRenamed(tag, f"{tag}_daily_max")
    daily_mean_pivot = daily_mean_pivot.withColumnRenamed(tag, f"{tag}_daily_mean")
    daily_std_pivot = daily_std_pivot.withColumnRenamed(tag, f"{tag}_daily_std")

# Join all pivots together
daily_features = daily_min_pivot \
    .join(daily_max_pivot, ["asset_id", "calendar_date"], "inner") \
    .join(daily_mean_pivot, ["asset_id", "calendar_date"], "inner") \
    .join(daily_std_pivot, ["asset_id", "calendar_date"], "inner")

print(f"   ✔ Daily features: {daily_features.count():,} rows × {len(daily_features.columns)} columns")

# 2. Add rolling 7-day and 14-day windows
print("\n2. Computing rolling trends (7d and 14d)...")

w_7d = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(-6, 0)
w_14d = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(-13, 0)

# For key tags, compute rolling averages and slopes
# (Use a subset of tags to keep feature count manageable)
KEY_TAGS = tag_list[:10]  # B4: first 10 alphabetically (deterministic after sort)

for tag in KEY_TAGS:
    mean_col = f"{tag}_daily_mean"
    
    if mean_col in daily_features.columns:
        # 7-day rolling average
        daily_features = daily_features.withColumn(
            f"{tag}_7d_avg",
            F.avg(mean_col).over(w_7d)
        )
        
        # 14-day rolling average
        daily_features = daily_features.withColumn(
            f"{tag}_14d_avg",
            F.avg(mean_col).over(w_14d)
        )
        
        # Degradation slope (7-day): linear regression slope
        # Simplified: (current - 7_days_ago) / 7
        daily_features = daily_features.withColumn(
            f"{tag}_7d_slope",
            (F.col(mean_col) - F.lag(mean_col, 7).over(Window.partitionBy("asset_id").orderBy("calendar_date"))) / 7.0
        )

print(f"   ✔ Added rolling trends for {len(KEY_TAGS)} key tags")
print(f"   ✔ Current feature count: {len(daily_features.columns)} columns")

# 3. Cumulative operating hours (days above threshold)
print("\n3. Computing cumulative operating hours...")

# Define "operating" as days where mean power/load > threshold
# (Adjust tag name based on your schema)
# B4: POWER_TAG requires per-asset load/power tag mapping. Verify units.
POWER_TAG = tag_list[0]
OPERATING_THRESHOLD = 50.0

if f"{POWER_TAG}_daily_mean" in daily_features.columns:
    daily_features = daily_features.withColumn(
        "is_operating",
        F.when(F.col(f"{POWER_TAG}_daily_mean") > OPERATING_THRESHOLD, 1).otherwise(0)
    )
    
    # Cumulative sum of operating days
    daily_features = daily_features.withColumn(
        "cumulative_operating_days",
        F.sum("is_operating").over(
            Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(Window.unboundedPreceding, 0)
        )
    )

# 4. Temporal features
print("\n4. Adding temporal features...")

daily_features = daily_features.withColumn(
    "day_of_week",
    F.dayofweek("calendar_date")  # 1=Sunday, 7=Saturday
).withColumn(
    "day_of_month",
    F.dayofmonth("calendar_date")
).withColumn(
    "month",
    F.month("calendar_date")
)

# Days since last stop (join with labels)
labels_for_days_since = spark.table("gold.daily_survival_labels").select(
    "asset_id",
    "calendar_date",
    F.when(F.col("days_to_next_stop") == 0, 1).otherwise(0).alias("is_stop_day")
)

# Self-join to find last stop date
w_lookback = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(Window.unboundedPreceding, -1)

labels_with_last_stop = labels_for_days_since.withColumn(
    "last_stop_date",
    F.when(F.col("is_stop_day") == 1, F.col("calendar_date")).otherwise(F.lit(None))
).withColumn(
    "last_stop_date",
    F.last("last_stop_date", ignorenulls=True).over(w_lookback)
).withColumn(
    "days_since_last_stop",
    F.when(F.col("last_stop_date").isNotNull(), 
           F.datediff(F.col("calendar_date"), F.col("last_stop_date"))
    ).otherwise(999)  # No prior stop
)

# Join back to features
daily_features = daily_features.join(
    labels_with_last_stop.select("asset_id", "calendar_date", "days_since_last_stop"),
    ["asset_id", "calendar_date"],
    "left"
)

print(f"   ✔ Added temporal features")
print(f"   ✔ Final feature count: {len(daily_features.columns)} columns")

# 5. Join with labels to create training dataset
print("\n5. Joining with labels...")

labels = spark.table("gold.daily_survival_labels")

training_data_daily = labels.join(
    daily_features,
    ["asset_id", "calendar_date"],
    "inner"
)

print(f"   ✔ Joined labels: {training_data_daily.count():,} rows")

# Drop rows with excessive nulls (early days without full history)
initial_count = training_data_daily.count()
training_data_daily = training_data_daily.dropna(subset=['days_to_next_stop'])  # Only drop rows missing the target label
final_count = training_data_daily.count()

print(f"   ✔ Dropped rows with >30% nulls: {initial_count - final_count:,} rows removed")
print(f"   ✔ Final training dataset: {final_count:,} rows × {len(training_data_daily.columns)} columns")

# Save to Delta table
training_data_daily.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ml.training_longterm")

print(f"\n✔ Saved ml.training_longterm")
print(f"\nSchema preview:")
training_data_daily.printSchema()

print(f"\nFeature summary (sample asset):")
sample_asset = TARGET_ASSETS[0]
training_data_daily.filter(F.col("asset_id") == sample_asset) \
    .orderBy(F.desc("calendar_date")) \
    .select("calendar_date", "days_to_next_stop", "label_stop_7d", "days_since_last_stop") \
    .show(20, truncate=False)

## Phase 2: Load Training Data

Load the same training dataset used in Phase-LongTerm-Survival, but prepare it for Cox modeling with proper duration and event columns.

In [ ]:
# Phase 2: Load and Prepare Data for Cox Model
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

print("="*70)
print("PHASE 2: LOADING TRAINING DATA")
print("="*70)

# Load training data from Phase 1b/2b above
training_df = spark.table("ml.training_longterm").toPandas()

print(f"\n✔ Loaded training data: {len(training_df):,} rows × {len(training_df.columns)} columns")
print(f"  Date range: {training_df['calendar_date'].min()} to {training_df['calendar_date'].max()}")

# Cox model requires:
# - duration: time until event or censoring (days_to_next_stop)
# - event: 1 if stop occurred, 0 if censored
training_df['duration'] = training_df['days_to_next_stop']
training_df['event'] = (~training_df['censored'].astype(bool)).astype(int)  # Invert: event=1 when NOT censored

print(f"\n✔ Cox model data prepared:")
print(f"  Duration column: days_to_next_stop (0-{HORIZON_DAYS})")
print(f"  Event column: 1=stop occurred, 0=censored (still running)")

# Event distribution
event_counts = training_df['event'].value_counts()
print(f"\n✔ Event distribution:")
print(f"  Stops (event=1): {event_counts.get(1, 0):,} ({event_counts.get(1, 0)/len(training_df)*100:.1f}%)")
print(f"  Censored (event=0): {event_counts.get(0, 0):,} ({event_counts.get(0, 0)/len(training_df)*100:.1f}%)")

# Temporal split with gap to prevent same-event leakage (B3 fix)
training_df = training_df.sort_values('calendar_date').reset_index(drop=True)
split_idx = int(len(training_df) * 0.8)
split_date = training_df.iloc[split_idx]['calendar_date']
gap_end = split_date + pd.Timedelta(days=HORIZON_DAYS)
train_df = training_df[training_df['calendar_date'] < split_date].copy()
test_df = training_df[training_df['calendar_date'] >= gap_end].copy()
print(f'  B3 gap: {HORIZON_DAYS}d buffer, train ends {split_date}, test starts {gap_end}')
print(f'  Gap removed {len(training_df) - len(train_df) - len(test_df):,} rows')

print(f"\n✔ Train/Test split (temporal):")
print(f"  Train: {len(train_df):,} rows ({train_df['calendar_date'].min()} to {train_df['calendar_date'].max()})")
print(f"    - Events: {train_df['event'].sum()}, Censored: {(~train_df['event'].astype(bool)).sum()}")
print(f"  Test:  {len(test_df):,} rows ({test_df['calendar_date'].min()} to {test_df['calendar_date'].max()})")
print(f"    - Events: {test_df['event'].sum()}, Censored: {(~test_df['event'].astype(bool)).sum()}")

# Prepare feature columns (same as regression model)
exclude_cols = ['asset_id', 'calendar_date', 'days_to_next_stop', 'censored',
                'label_stop_7d', 'label_stop_14d', 'duration', 'event',
                'days_since_last_stop',  # B2: derived from labels, leaks target
                'is_operating', 'cumulative_operating_days',  # B4: unreliable
                'is_stop_day', 'last_stop_date']  # B2: label-derived

# Exclude spurious calendar features (day_of_week, day_of_month) from candidate pool.
# These have no causal relationship with equipment failure; when they rank highly
# (e.g., day_of_month showing |C-0.5|=0.14 for BFP East), it indicates the model is
# memorizing temporal patterns in a small dataset rather than learning physical
# degradation signals. We keep 'month' (seasonality is physically meaningful for
# thermal load) and 'days_since_last_stop' (physical recovery-time meaning).
SPURIOUS_TEMPORAL = ['day_of_week', 'day_of_month']
feature_cols = [c for c in training_df.columns
                if c not in exclude_cols and c not in SPURIOUS_TEMPORAL]

print(f"\n✔ Feature matrix: {len(feature_cols)} features")
print(f"  Excluded calendar features (no causal link to failure): {SPURIOUS_TEMPORAL}")
print(f"  Sample features: {feature_cols[:5]}")

## Phase 3: Train Cox Proportional Hazards Model

Train the Cox model using the `lifelines` library. The model learns how each feature affects the **hazard** (instantaneous risk of stop).

In [ ]:
# Phase 3: Train PER-ASSET Cox Proportional Hazards Models
#
# The original pooled model trained ONE Cox on all 3 assets together. Because
# feature columns are asset-disjoint (RV2 tags are null/zero on RV3 rows and
# vice-versa), the coefficients collapsed toward 0 (max |coef| ~0.01) and the
# pooled CV C-index ran ~0.45 (below random). We now train a SEPARATE Cox model
# per asset using only that asset's non-zero-variance tags.
#
# WARNING: RV3_U3_Steam_Turbine and RV3_U3_Boiler_Feed_Pump_East live on the
# same physical unit, so their event labels are identical. Each model still uses
# its own subsystem-specific tags; risk rankings may track closely, but the
# driver explanations will differ by which tag pool the model actually saw.
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import mlflow
import mlflow.sklearn
import numpy as np

print("="*70)
print("PHASE 3: TRAINING PER-ASSET COX PH MODELS")
print("="*70)

mlflow.set_experiment(EXPERIMENT_NAME)

models = {}              # asset_id -> {cph, features, scaler, penalizer, cv/train/test ci, run_id}
per_asset_features = {}  # asset_id -> list[str]

PENALIZERS = [0.5, 1.0, 2.0, 5.0, 10.0]

trained_assets = train_df['asset_id'].unique().tolist()
sorted_assets = [a for a in TARGET_ASSETS if a in trained_assets]

for asset in sorted_assets:
    print("\n" + "="*70)
    print(f"ASSET: {asset}")
    print("="*70)

    asset_train = (train_df[train_df['asset_id'] == asset]
                   .copy().sort_values('calendar_date').reset_index(drop=True))
    asset_test  = (test_df[test_df['asset_id'] == asset]
                   .copy().sort_values('calendar_date').reset_index(drop=True))

    n_events_train = int(asset_train['event'].sum())
    n_events_test  = int(asset_test['event'].sum())
    print(f"  Train: {len(asset_train):,} rows, {n_events_train} events")
    print(f"  Test : {len(asset_test):,} rows, {n_events_test} events")

    if n_events_train < 5:
        print(f"  ⚠️  Skipping — fewer than 5 training events.")
        continue

    # 1) Candidate features: non-zero variance on THIS asset only
    asset_candidates = []
    for c in feature_cols:
        if c in exclude_cols:
            continue
        col = asset_train[c]
        if col.isna().all():
            continue
        if col.fillna(0).std() == 0:
            continue
        asset_candidates.append(c)
    print(f"  Candidate features (non-zero var on this asset): {len(asset_candidates)}")
    if len(asset_candidates) < 2:
        print(f"  ⚠️  Too few candidate features — skipping.")
        continue

    # 2) Univariate concordance ranking
    feature_scores = {}
    for col in asset_candidates:
        try:
            vals = asset_train[col].fillna(0)
            ci = concordance_index(asset_train['duration'], -vals, asset_train['event'])
            feature_scores[col] = abs(ci - 0.5)
        except Exception:
            continue
    ranked = sorted(feature_scores.items(), key=lambda x: x[1], reverse=True)

    # 3) EPV-based cap: events / 10, floor 3, ceiling 20
    max_features = max(3, n_events_train // 10)
    max_features = min(max_features, 20)
    print(f"  EPV cap: events({n_events_train})//10 = {n_events_train//10} → up to {max_features} features")

    # 4) Greedy dedup at |corr| > 0.9
    selected = []
    for feat, _score in ranked:
        if len(selected) >= max_features:
            break
        redundant = False
        for existing in selected:
            try:
                corr = abs(asset_train[feat].fillna(0).corr(asset_train[existing].fillna(0)))
                if pd.notna(corr) and corr > 0.9:
                    redundant = True
                    break
            except Exception:
                pass
        if not redundant:
            selected.append(feat)

    if len(selected) < 2:
        print(f"  ⚠️  Only {len(selected)} feature(s) survived dedup — skipping.")
        continue
    print(f"  Selected {len(selected)} features after dedup (top by |C-0.5|):")
    for f in selected:
        print(f"    {f}: |C-0.5| = {feature_scores[f]:.4f}")

    # 5) Standardize on THIS asset's training data
    asset_scaler = StandardScaler()
    Xtr_raw = (asset_train[selected].fillna(0)
                .replace([np.inf, -np.inf], 0))
    Xtr_s = pd.DataFrame(asset_scaler.fit_transform(Xtr_raw),
                         columns=selected, index=asset_train.index).clip(-5, 5)
    cox_train = Xtr_s.copy()
    cox_train['duration'] = asset_train['duration'].values
    cox_train['event']    = asset_train['event'].values

    cox_test = None
    if len(asset_test) > 0:
        Xte_raw = (asset_test[selected].fillna(0)
                    .replace([np.inf, -np.inf], 0))
        Xte_s = pd.DataFrame(asset_scaler.transform(Xte_raw),
                             columns=selected, index=asset_test.index).clip(-5, 5)
        cox_test = Xte_s.copy()
        cox_test['duration'] = asset_test['duration'].values
        cox_test['event']    = asset_test['event'].values

    # 6) Per-asset penalizer tuning via walk-forward CV
    n_splits = min(5, max(2, n_events_train // 10))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_p, best_cv = PENALIZERS[0], -np.inf
    print(f"  Penalizer tuning ({n_splits}-fold walk-forward CV):")
    for p in PENALIZERS:
        scores = []
        for tr_idx, va_idx in tscv.split(cox_train):
            fold_tr = cox_train.iloc[tr_idx]
            fold_va = cox_train.iloc[va_idx]
            if fold_tr['event'].sum() < 2 or fold_va['event'].sum() < 1:
                continue
            try:
                cph_cv = CoxPHFitter(penalizer=p, l1_ratio=0.0)
                cph_cv.fit(fold_tr, duration_col='duration', event_col='event', show_progress=False)
                ci = concordance_index(
                    fold_va['duration'],
                    -cph_cv.predict_partial_hazard(fold_va[selected]),
                    fold_va['event'],
                )
                scores.append(ci)
            except Exception:
                continue
        if scores:
            mean_ci = float(np.mean(scores))
            marker = " ← best" if mean_ci > best_cv else ""
            print(f"    penalizer={p:>5.2f}: CV C-index = {mean_ci:.4f} (n_folds={len(scores)}){marker}")
            if mean_ci > best_cv:
                best_cv = mean_ci
                best_p = p
        else:
            print(f"    penalizer={p:>5.2f}: no usable folds")
    print(f"  → Best penalizer: {best_p} (CV C-index = {best_cv:.4f})")

    # 7) Fit final per-asset Cox
    cph_a = CoxPHFitter(penalizer=best_p, l1_ratio=0.0)
    try:
        cph_a.fit(cox_train, duration_col='duration', event_col='event', show_progress=False)
    except Exception as e:
        print(f"  ⚠️  Cox fit failed: {e}")
        continue

    train_ci = float(cph_a.concordance_index_)
    test_ci = None
    if cox_test is not None and cox_test['event'].sum() >= 1:
        try:
            test_ci = float(concordance_index(
                cox_test['duration'],
                -cph_a.predict_partial_hazard(cox_test[selected]),
                cox_test['event'],
            ))
        except Exception as e:
            print(f"  ⚠️  Test C-index failed: {e}")

    gap = (train_ci - test_ci) if test_ci is not None else None
    print(f"  Train C-index: {train_ci:.4f}")
    if test_ci is not None:
        print(f"  Test  C-index: {test_ci:.4f}  (train-test gap {gap:+.4f})")
    if best_cv < 0.55:
        print(f"  ⚠️  CV C-index {best_cv:.4f} < 0.55 — model is weak. Treat outputs as advisory;")
        print(f"      with this much data the features are the bottleneck, not the algorithm.")

    # 8) MLflow logging per asset
    asset_run_id = None
    try:
        with mlflow.start_run(run_name=f"Cox-{asset}-14d"):
            mlflow.log_param("model_type", "CoxPH_per_asset")
            mlflow.log_param("asset_id", asset)
            mlflow.log_param("horizon", "14_days")
            mlflow.log_param("n_features", len(selected))
            mlflow.log_param("n_train_rows", len(cox_train))
            mlflow.log_param("n_train_events", n_events_train)
            mlflow.log_param("n_test_rows", len(cox_test) if cox_test is not None else 0)
            mlflow.log_param("n_test_events", n_events_test)
            mlflow.log_param("penalizer", best_p)
            mlflow.log_param("cv_method", "TimeSeriesSplit")
            mlflow.log_param("cv_splits", n_splits)
            mlflow.log_metric("cv_c_index", best_cv)
            mlflow.log_metric("train_c_index", train_ci)
            if test_ci is not None:
                mlflow.log_metric("test_c_index", test_ci)
                mlflow.log_metric("train_test_gap", gap)
            asset_run_id = mlflow.active_run().info.run_id
    except Exception as e:
        print(f"  ⚠️  MLflow logging failed: {e}")

    # 9) Top coefficients
    fi = pd.DataFrame({
        'feature': cph_a.params_.index,
        'coefficient': cph_a.params_.values,
        'hazard_ratio': np.exp(cph_a.params_.values),
    })
    fi['abs_coef'] = fi['coefficient'].abs()
    fi = fi.sort_values('abs_coef', ascending=False)
    print(f"  Top {min(10, len(fi))} features by |coef|:")
    print(f"    {'feature':<45} {'coef':>9} {'HR':>8}  dir")
    for _, r in fi.head(10).iterrows():
        arrow = "↑ risk" if r['coefficient'] > 0 else "↓ risk"
        print(f"    {r['feature']:<45} {r['coefficient']:>+9.4f} {r['hazard_ratio']:>8.3f}  {arrow}")

    models[asset] = {
        'cph': cph_a,
        'features': selected,
        'scaler': asset_scaler,
        'penalizer': best_p,
        'cv_c_index': best_cv,
        'train_c_index': train_ci,
        'test_c_index': test_ci,
        'run_id': asset_run_id,
    }
    per_asset_features[asset] = selected

print("\n" + "="*70)
print("PER-ASSET TRAINING SUMMARY")
print("="*70)
print(f"  {'Asset':<35} {'Feat':>5} {'CV CI':>8} {'TrainCI':>9} {'TestCI':>9} {'Pen':>5}")
print(f"  {'-'*78}")
for a in sorted_assets:
    if a not in models:
        print(f"  {a:<35} (skipped — insufficient data)")
        continue
    m = models[a]
    test_str = f"{m['test_c_index']:.4f}" if m['test_c_index'] is not None else "   n/a"
    print(f"  {a:<35} {len(m['features']):>5d} {m['cv_c_index']:>8.4f} {m['train_c_index']:>9.4f} {test_str:>9} {m['penalizer']:>5.2f}")

if not models:
    raise RuntimeError("No per-asset Cox models were trained — check feature/event data.")

# Backwards-compat shims for any leftover diagnostic code. Phase 4/5 below use
# the per-asset `models` dict directly.
_first_asset = next(a for a in sorted_assets if a in models)
cph = models[_first_asset]['cph']
scaler = models[_first_asset]['scaler']
top_features = models[_first_asset]['features']
selected_features = top_features
best_penalizer = models[_first_asset]['penalizer']
test_c_index = (models[_first_asset]['test_c_index']
                if models[_first_asset]['test_c_index'] is not None else float('nan'))
run_id = models[_first_asset]['run_id']

print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")


## Alternative: Random Survival Forest

Train a Random Survival Forest (non-parametric) to compare against Cox PH. RSF doesn't require the proportional hazards assumption and can capture non-linear feature interactions.

In [ ]:
# Random Survival Forest — per-asset alternative to Cox PH
#
# Like the Cox training above, we now fit ONE RSF per asset using only that
# asset's tag pool. We keep the aggressive regularization from the prior
# pooled run because per-asset row counts are small.
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sklearn.inspection import permutation_importance
import numpy as np

print("="*70)
print("RANDOM SURVIVAL FOREST (per-asset)")
print("="*70)

rsf_results = {}

for asset in sorted_assets:
    if asset not in models:
        continue
    print(f"\n--- {asset} ---")

    asset_train = train_df[train_df['asset_id'] == asset].copy()
    asset_test  = test_df[test_df['asset_id'] == asset].copy()
    feats = models[asset]['features']

    if len(asset_train) < 30 or asset_train['event'].sum() < 5:
        print("  Too few rows/events — skipping RSF for this asset.")
        continue

    Xtr = asset_train[feats].fillna(0).replace([np.inf, -np.inf], 0).values
    ytr = np.array([(bool(e), d) for e, d in zip(asset_train['event'], asset_train['duration'])],
                   dtype=[('event', bool), ('duration', float)])

    Xte = yte = None
    if len(asset_test) > 0:
        Xte = asset_test[feats].fillna(0).replace([np.inf, -np.inf], 0).values
        yte = np.array([(bool(e), d) for e, d in zip(asset_test['event'], asset_test['duration'])],
                       dtype=[('event', bool), ('duration', float)])

    rsf = RandomSurvivalForest(
        n_estimators=200,
        min_samples_leaf=50,
        min_samples_split=100,
        max_features="sqrt",
        max_depth=3,
        random_state=42,
        n_jobs=-1,
    )
    try:
        rsf.fit(Xtr, ytr)
    except Exception as e:
        print(f"  RSF fit failed: {e}")
        continue

    rsf_tr_ci = float(concordance_index_censored(ytr['event'], ytr['duration'], rsf.predict(Xtr))[0])
    rsf_te_ci = None
    if Xte is not None and yte['event'].sum() > 0:
        rsf_te_ci = float(concordance_index_censored(yte['event'], yte['duration'], rsf.predict(Xte))[0])

    print(f"  RSF train C-index: {rsf_tr_ci:.4f}")
    if rsf_te_ci is not None:
        gap = rsf_tr_ci - rsf_te_ci
        print(f"  RSF test  C-index: {rsf_te_ci:.4f}  (train-test gap {gap:+.4f})")
        if rsf_te_ci < 0.55:
            print(f"  ⚠️  RSF failed to generalize (test CI < 0.55).")

    cox_te = models[asset]['test_c_index']
    if cox_te is not None and rsf_te_ci is not None:
        winner = "RSF" if rsf_te_ci > cox_te else "Cox"
        print(f"  Cox test C-index:  {cox_te:.4f}  → {winner} wins on this asset")

    rsf_results[asset] = {'rsf': rsf, 'train_ci': rsf_tr_ci, 'test_ci': rsf_te_ci}

    try:
        with mlflow.start_run(run_name=f"RSF-{asset}-14d"):
            mlflow.log_param("model_type", "RSF_per_asset")
            mlflow.log_param("asset_id", asset)
            mlflow.log_param("n_estimators", 200)
            mlflow.log_param("min_samples_leaf", 50)
            mlflow.log_param("min_samples_split", 100)
            mlflow.log_param("max_depth", 3)
            mlflow.log_param("n_features", len(feats))
            mlflow.log_metric("train_concordance", rsf_tr_ci)
            if rsf_te_ci is not None:
                mlflow.log_metric("test_concordance", rsf_te_ci)
    except Exception as e:
        print(f"  ⚠️  MLflow logging failed: {e}")

# Head-to-head summary
cox_wins = rsf_wins = 0
for a, r in rsf_results.items():
    cox_te = models[a]['test_c_index']
    if r['test_ci'] is not None and cox_te is not None:
        if r['test_ci'] > cox_te:
            rsf_wins += 1
        else:
            cox_wins += 1

print(f"\nHead-to-head test C-index: Cox wins {cox_wins} / RSF wins {rsf_wins}")
best_model_type = "Cox"  # downstream scoring always uses the per-asset Cox models
print(f"Downstream scoring uses: {best_model_type} (per-asset)")


## Phase 4: Generate Predictions

Unlike regression, Cox models output:
1. **Risk Score** (relative hazard) - higher = more likely to stop soon
2. **Survival Probability** at specific time points - P(survive > t days)

We'll score the latest available data and save predictions with interpretable risk levels.

In [ ]:
# Phase 4: Generate Survival Predictions (per-asset)
from datetime import datetime

print("="*70)
print("PHASE 4: GENERATING SURVIVAL PREDICTIONS")
print("="*70)

training_data = spark.table("ml.training_longterm")
latest_date = training_data.agg(F.max("calendar_date")).collect()[0][0]
print(f"\n✔ Latest available data: {latest_date}")

scoring_data = training_data.filter(F.col("calendar_date") == latest_date).toPandas()
# Filter to only ACTIVE (running) assets
scoring_data = scoring_data[scoring_data['asset_id'].isin(ACTIVE_ASSETS)]
print(f"  Assets to score: {len(scoring_data)}")
print(f"  Assets: {scoring_data['asset_id'].tolist()}")
if STOPPED_ASSETS:
    print(f"  ⚠️ Skipped (not running): {', '.join(STOPPED_ASSETS)}")

scoring_results = []
for _, srow in scoring_data.iterrows():
    asset = srow['asset_id']
    if asset not in models:
        print(f"  {asset}: no per-asset model trained — skipping")
        continue
    m = models[asset]
    feats = m['features']

    raw = pd.DataFrame(
        [[float(srow[f]) if pd.notna(srow[f]) else 0.0 for f in feats]],
        columns=feats,
    ).replace([np.inf, -np.inf], 0).fillna(0)
    scaled = pd.DataFrame(m['scaler'].transform(raw), columns=feats).clip(-5, 5)

    risk    = float(m['cph'].predict_partial_hazard(scaled).values[0])
    surv_7  = float(m['cph'].predict_survival_function(scaled, times=[7]).T.values.flatten()[0])
    surv_14 = float(m['cph'].predict_survival_function(scaled, times=[14]).T.values.flatten()[0])
    med_result = m['cph'].predict_median(scaled)
    med_raw = float(med_result.values[0]) if hasattr(med_result, 'values') else float(med_result)
    med = 999.0 if np.isinf(med_raw) else float(med_raw)

    if surv_14 < 0.30:
        risk_level = "CRITICAL"
    elif surv_14 < 0.50:
        risk_level = "HIGH"
    elif surv_14 < 0.70:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    scoring_results.append({
        'scoring_timestamp': datetime.now(),
        'scoring_date': latest_date,
        'asset_id': asset,
        'risk_score': risk,
        'survival_probability_7d': surv_7,
        'survival_probability_14d': surv_14,
        'predicted_median_survival_days': med,
        'risk_level': risk_level,
        'model_type': 'cox_survival_per_asset',
        'horizon': '14_days',
        'model_run_id': m['run_id'] or 'NA',
    })

print(f"\n✔ Predictions (each uses its asset-specific model):")
print(f"\n  {'Asset':<35} {'Risk':>8} {'7d Surv':>9} {'14d Surv':>9} {'Median':>8}")
print(f"  {'-'*80}")
for r in scoring_results:
    md = "999+ d" if r['predicted_median_survival_days'] >= 999 else f"{r['predicted_median_survival_days']:5.1f} d"
    print(f"  {r['asset_id']:<35} {r['risk_score']:8.3f} {r['survival_probability_7d']:8.1%} {r['survival_probability_14d']:8.1%} {md:>8}")

print(f"\n✔ Interpretation:")
print(f"  - Risk Score: Relative hazard from this asset's own Cox model (not comparable across assets)")
print(f"  - 7d/14d Survival: P(survive without stop)")
print(f"  - Median: Days until 50% probability of stop (999 = survival never drops below 50%)")

scoring_results_df = pd.DataFrame(scoring_results)
print(f"\n✔ Risk Level Distribution:")
print(scoring_results_df['risk_level'].value_counts().to_string())

scoring_results_spark = spark.createDataFrame(scoring_results_df)
from datetime import datetime as _dt
scoring_results_spark = scoring_results_spark.withColumn("model_run_timestamp", F.lit(_dt.utcnow()))
scoring_results_spark.write.mode("append").format("delta").saveAsTable("ml.predictions_longterm")
print(f"\n✔ Saved predictions to ml.predictions_longterm")

print(f"\n{'='*70}")
print("SCORING COMPLETE")
print(f"{'='*70}")

print(f"\n📊 Per-asset 14-day survival (should now differ meaningfully across assets):")
for _, row in scoring_results_df.iterrows():
    print(f"    {row['asset_id']}: {row['survival_probability_14d']:.1%} survival → {row['risk_level']}")
try:
    reg_preds = spark.table("ml.predictions_longterm").toPandas()
    latest_reg = reg_preds.sort_values("scoring_date").groupby("asset_id").last().reset_index()
    print(f"\n  Regression Model Output (latest scores):")
    for _, rrow in latest_reg.iterrows():
        print(f"    {rrow['asset_id']}: {rrow.get('predicted_days_to_stop', 'N/A')} predicted days")
except Exception:
    print(f"\n  ⚠️ Regression predictions table not found — run Phase-LongTerm-Survival to compare")

print(f"\n💡 Key Difference:")
print(f"   Cox model provides probabilistic survival curve (handles censoring)")
print(f"   Regression provides point estimate (biased toward censoring limit)")


## Phase 5: Prediction Drivers Analysis

For each asset, decompose the risk score into individual feature contributions to answer **"Why does the model predict this risk level?"**

- **Contribution** = Coefficient × Feature Value (in the linear predictor)
- Positive contributions push hazard UP (increase risk)
- Negative contributions push hazard DOWN (decrease risk)
- The sum of all contributions determines the asset's overall risk score

In [ ]:
# Phase 5: Prediction Drivers Analysis (per-asset)
import re

print("="*70)
print("PHASE 5: PREDICTION DRIVERS ANALYSIS")
print("="*70)

# Load PI tag metadata for descriptors
try:
    tag_meta = spark.table("dbo.pi_tags_metadata").select("Name", "Descriptor", "EngineeringUnits").toPandas()
    tag_meta['tag_name_normalized'] = (tag_meta['Name']
        .str.replace(':', '_', regex=False)
        .str.replace('.', '_', regex=False))
    tag_lookup = tag_meta.set_index('tag_name_normalized')[['Descriptor', 'EngineeringUnits']].to_dict('index')
    print(f"✔ Loaded {len(tag_meta)} tag descriptors from dbo.pi_tags_metadata")
except Exception as e:
    print(f"⚠️ Could not load dbo.pi_tags_metadata: {e}")
    tag_lookup = {}

def extract_tag_info(feature_name):
    suffix_pattern = r'_(daily_min|daily_max|daily_mean|daily_std)$'
    match = re.search(suffix_pattern, feature_name)
    if match:
        stat_type = match.group(1)
        tag_name = feature_name[:match.start()]
    else:
        stat_type = "unknown"
        tag_name = feature_name
    meta = tag_lookup.get(tag_name, {})
    return tag_name, stat_type, meta.get('Descriptor', ''), meta.get('EngineeringUnits', '')

all_driver_records = []

for _, row in scoring_results_df.iterrows():
    asset_id = row['asset_id']
    if asset_id not in models:
        continue
    m = models[asset_id]
    feats = m['features']
    coefs = m['cph'].params_

    asset_row_df = scoring_data[scoring_data['asset_id'] == asset_id]
    if asset_row_df.empty:
        continue
    asset_row = asset_row_df.iloc[0]

    print(f"\n{'='*70}")
    print(f"ASSET: {asset_id}")
    print(f"Risk Score: {row['risk_score']:.3f} | Risk Level: {row['risk_level']}")
    print(f"14-day Survival: {row['survival_probability_14d']:.1%}")
    print(f"Model features: {len(feats)} (unique to this asset; coefficients come from its own Cox fit)")
    print(f"{'='*70}")

    raw_vals = pd.DataFrame(
        [[float(asset_row[f]) if pd.notna(asset_row[f]) else 0.0 for f in feats]],
        columns=feats,
    ).replace([np.inf, -np.inf], 0).fillna(0)
    scaled_vals = m['scaler'].transform(raw_vals)[0]
    scaled_vals = np.clip(scaled_vals, -5, 5)

    contributions = []
    for i, feat in enumerate(feats):
        coef = float(coefs.get(feat, 0.0))
        hr = float(np.exp(coef))
        value = float(asset_row[feat]) if pd.notna(asset_row[feat]) else 0.0
        contribution = coef * scaled_vals[i]
        tag_name, stat_type, descriptor, units = extract_tag_info(feat)
        contributions.append({
            'feature': feat, 'tag_name': tag_name, 'stat_type': stat_type,
            'descriptor': descriptor, 'units': units,
            'value': value, 'coefficient': coef, 'hazard_ratio': hr,
            'contribution': contribution, 'abs_contribution': abs(contribution),
        })

    contrib_df = pd.DataFrame(contributions).sort_values('abs_contribution', ascending=False)

    top_inc = contrib_df[contrib_df['contribution'] > 0].head(10)
    if len(top_inc) > 0:
        print(f"\n🔴 TOP RISK INCREASERS (driving hazard UP):")
        print(f"  {'Feature':<42} {'Descriptor':<30} {'Value':>8} {'Contrib':>10}")
        print(f"  {'-'*95}")
        for _, r in top_inc.iterrows():
            desc = (r['descriptor'] or '')[:28]
            us = f" {r['units']}" if r['units'] else ''
            print(f"  {r['feature']:<42} {desc:<30} {r['value']:>8.2f}{us} {r['contribution']:>+10.4f}")

    top_dec = contrib_df[contrib_df['contribution'] < 0].sort_values('contribution').head(10)
    if len(top_dec) > 0:
        print(f"\n🟢 TOP RISK REDUCERS (driving hazard DOWN):")
        print(f"  {'Feature':<42} {'Descriptor':<30} {'Value':>8} {'Contrib':>10}")
        print(f"  {'-'*95}")
        for _, r in top_dec.iterrows():
            desc = (r['descriptor'] or '')[:28]
            us = f" {r['units']}" if r['units'] else ''
            print(f"  {r['feature']:<42} {desc:<30} {r['value']:>8.2f}{us} {r['contribution']:>+10.4f}")

    tot_p = contrib_df[contrib_df['contribution'] > 0]['contribution'].sum()
    tot_n = contrib_df[contrib_df['contribution'] < 0]['contribution'].sum()
    print(f"\n  📊 Net Contribution Summary:")
    print(f"     Risk-increasing contributions: {tot_p:>+10.4f}")
    print(f"     Risk-reducing contributions:   {tot_n:>+10.4f}")
    print(f"     Net linear predictor:          {tot_p+tot_n:>+10.4f}")

    for rank, (_, r) in enumerate(contrib_df.head(10).iterrows(), 1):
        direction = "risk_increaser" if r['contribution'] > 0 else "risk_reducer"
        all_driver_records.append({
            'scoring_date': row['scoring_date'],
            'asset_id': asset_id,
            'risk_level': row['risk_level'],
            'risk_score': float(row['risk_score']),
            'feature': r['feature'],
            'tag_name': r['tag_name'],
            'stat_type': r['stat_type'],
            'descriptor': r['descriptor'] or None,
            'engineering_units': r['units'] or None,
            'feature_value': float(r['value']),
            'coefficient': float(r['coefficient']),
            'hazard_ratio': float(r['hazard_ratio']),
            'contribution': float(r['contribution']),
            'driver_rank': rank,
            'driver_direction': direction,
            'model_run_id': m['run_id'] or 'NA',
        })

drivers_df = pd.DataFrame(all_driver_records)
drivers_spark = spark.createDataFrame(drivers_df)
from datetime import datetime
model_run_timestamp = datetime.utcnow()
drivers_spark = drivers_spark.withColumn("model_run_timestamp", F.lit(model_run_timestamp))
drivers_spark.write.mode("append").format("delta").saveAsTable("ml.drivers_longterm")

print(f"\n{'='*70}")
print(f"✔ Saved {len(all_driver_records)} driver records to ml.drivers_longterm")
print(f"  (Top 10 drivers per asset × {len(scoring_results_df)} assets, from per-asset Cox models)")
print(f"{'='*70}")


## Phase 6: Actionable Watch Recommendations

For each top risk driver, generate contextual recommendations by comparing the current
sensor value against its 30-day baseline (mean ± 2σ) and 7-day trend.

**Output:** Natural-language watch items per asset with:
- Current value vs normal range
- Trend direction and rate of change
- What to watch for over the next 7–14 days
- Recommended action (monitor / investigate / act)


In [ ]:
# Phase 6: Simplified watchlist - 1 row per asset with 14d probability + top drivers
from datetime import datetime
import uuid

WATCH_HORIZON_DAYS = 14
notebook_run_id = str(uuid.uuid4())

print("="*70)
print("WATCHLIST SUMMARY")
print("="*70)

if len(all_driver_records) == 0 or len(scoring_results) == 0:
    print("No driver/prediction records - run Phase 4 & 5 first")
else:
    drivers_pdf = pd.DataFrame(all_driver_records)
    
    # Load friendly tag names from bridge table
    bridge_desc = spark.table("gold.bridge_pi_tag_to_asset") \
        .select("Tag", "tag_description").dropDuplicates(["Tag"]).toPandas()
    tag_desc_map = dict(zip(
        bridge_desc['Tag'].str.replace(':', '_').str.replace('.', '_'),
        bridge_desc['tag_description']
    ))
    
    watch_records = []
    for pred in scoring_results:
        asset = pred['asset_id']
        surv_14 = pred['survival_probability_14d']
        downtime_prob = (1 - surv_14) * 100  # Convert to % probability of downtime
        risk_level = pred['risk_level']
        
        # Get top 3 risk-increasing drivers for this asset
        ad = drivers_pdf[
            (drivers_pdf['asset_id'] == asset) &
            (drivers_pdf['driver_direction'] == 'risk_increaser')
        ].sort_values('driver_rank').head(3)
        
        if len(ad) == 0:
            continue
        
        # Build friendly names for top drivers
        friendly = []
        for _, d in ad.iterrows():
            tn = d['tag_name']
            desc = tag_desc_map.get(tn, d.get('descriptor') or tn)
            friendly.append(desc.title() if desc else tn)
        watch_str = ", ".join(dict.fromkeys(friendly))
        
        top_tag = ad.iloc[0]['tag_name']
        top_contribution = float(ad.iloc[0]['contribution'])
        
        rec_text = f"{downtime_prob:.0f}% predicted downtime probability over 14 days. Watch {watch_str}."
        
        watch_records.append({
            'model_name': 'CoxPH_LongTerm',
            'scoring_date': datetime.utcnow().strftime('%Y-%m-%d'),
            'asset_id': asset,
            'feature': 'cox_summary',
            'tag_name': safe_to_original.get(top_tag, top_tag),
            'descriptor': friendly[0] if friendly else None,
            'engineering_units': None,
            'current_value': downtime_prob / 100,
            'baseline_mean': None,
            'baseline_std': None,
            'normal_range_low': None,
            'normal_range_high': None,
            'risk_contribution': top_contribution,
            'trend_direction': None,
            'trend_slope_per_day': None,
            'recommended_action': risk_level,
            'recommendation_text': rec_text,
            'watch_horizon_days': WATCH_HORIZON_DAYS,
            'model_run_timestamp': datetime.utcnow(),
            'notebook_run_id': notebook_run_id
        })
        
        print(f"  {asset}: {rec_text}")
    
    if watch_records:
        from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
        schema = StructType([
            StructField('model_name', StringType()), StructField('scoring_date', StringType()),
            StructField('asset_id', StringType()), StructField('feature', StringType()),
            StructField('tag_name', StringType()), StructField('descriptor', StringType()),
            StructField('engineering_units', StringType()), StructField('current_value', DoubleType()),
            StructField('baseline_mean', DoubleType()), StructField('baseline_std', DoubleType()),
            StructField('normal_range_low', DoubleType()), StructField('normal_range_high', DoubleType()),
            StructField('risk_contribution', DoubleType()), StructField('trend_direction', StringType()),
            StructField('trend_slope_per_day', DoubleType()), StructField('recommended_action', StringType()),
            StructField('recommendation_text', StringType()), StructField('watch_horizon_days', IntegerType()),
            StructField('model_run_timestamp', TimestampType()), StructField('notebook_run_id', StringType())
        ])
        wl_df = spark.createDataFrame(pd.DataFrame(watch_records), schema=schema)
        wl_df.write.mode('append').option('mergeSchema', 'true').format('delta').saveAsTable('ml.watchlist')
        print(f"\nSaved {len(watch_records)} watchlist entries")
    else:
        print("No watch entries to save")